# PVT + ADX Strategy (Advanced)

Evaluation of Price Volume Trend combined with ADX
across multiple assets and timeframes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from strategies.advanced.pvt_adx import PVTADXStrategy

In [ ]:
TRANSACTION_COST = 0.0015


def evaluate_strategy(df):
    df = df.copy()

    df["returns"] = df["close"].pct_change()
    df["strategy_returns"] = df["signal"].shift(1) * df["returns"]

    df["strategy_returns"] -= np.where(
        df["signal"] != 0, TRANSACTION_COST, 0
    )

    df["cum_market"] = (1 + df["returns"]).cumprod()
    df["cum_strategy"] = (1 + df["strategy_returns"]).cumprod()
    df["drawdown"] = df["cum_strategy"] / df["cum_strategy"].cummax() - 1

    return df


def plot_results(df, title):
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

    ax1.plot(df.index, df["strategy_returns"])
    ax1.set_title(f"{title} – Strategy Returns")

    ax2.plot(df.index, df["cum_market"], label="Market")
    ax2.plot(df.index, df["cum_strategy"], label="Strategy")
    ax2.legend()
    ax2.set_title(f"{title} – Cumulative Returns")

    ax3.plot(df.index, df["drawdown"], color="red")
    ax3.set_title(f"{title} – Drawdown")

    plt.tight_layout()
    plt.show()


In [ ]:
def run_experiment(csv_path, title, ema_length, adx_threshold):
    df = pd.read_csv(csv_path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df.set_index("datetime", inplace=True)

    strategy = PVTADXStrategy(
        ema_length=ema_length,
        adx_threshold=adx_threshold,
    )

    df = strategy.compute_indicators(df)
    df = strategy.generate_signals(df)
    df = evaluate_strategy(df)

    plot_results(df, title)
    return df

In [ ]:
# BTC
btc_1d = run_experiment(
    "data/processed/btc_1d.csv",
    "BTC 1D – PVT + ADX",
    ema_length=200,
    adx_threshold=15,
)

btc_4h = run_experiment(
    "data/processed/btc_4h.csv",
    "BTC 4H – PVT + ADX",
    ema_length=200,
    adx_threshold=15,
)

# ETH
eth_1d = run_experiment(
    "data/processed/eth_1d.csv",
    "ETH 1D – PVT + ADX",
    ema_length=200,
    adx_threshold=15,
)

eth_4h = run_experiment(
    "data/processed/eth_4h.csv",
    "ETH 4H – PVT + ADX",
    ema_length=200,
    adx_threshold=15,
)

In [ ]:
import os

def save_experiment_summary(
    df,
    strategy_name,
    asset,
    timeframe,
    out_dir="results/csv_outputs"
):
    """
    Save a one-row CSV summarizing strategy performance.
    """

    os.makedirs(out_dir, exist_ok=True)

    total_return = df["cumulative_strategy_returns"].iloc[-1] - 1
    benchmark_return = df["cumulative_market_returns"].iloc[-1] - 1
    max_drawdown = df["drawdown"].min()
    win_rate = (df["strategy_returns"] > 0).mean()
    total_trades = df["signal"].diff().abs().sum() / 2

    row = {
        "strategy": strategy_name,
        "asset": asset,
        "timeframe": timeframe,
        "total_return_pct": round(total_return * 100, 2),
        "benchmark_return_pct": round(benchmark_return * 100, 2),
        "max_drawdown_pct": round(max_drawdown * 100, 2),
        "win_rate_pct": round(win_rate * 100, 2),
        "total_trades": int(total_trades),
    }

    df_out = pd.DataFrame([row])

    out_file = os.path.join(out_dir, f"{strategy_name}.csv")

    if os.path.exists(out_file):
        df_out.to_csv(out_file, mode="a", header=False, index=False)
    else:
        df_out.to_csv(out_file, index=False)

    print(f"✅ Saved summary → {out_file}")


In [ ]:
save_experiment_summary(
    df=btc,
    strategy_name="pvt_mdx",
    asset="BTC",
    timeframe="1D"
)

In [ ]:
save_experiment_summary(
    df=btc,
    strategy_name="pvt_mdx",
    asset="BTC",
    timeframe="4H"
)

In [ ]:
save_experiment_summary(
    df=btc,
    strategy_name="pvt_mdx",
    asset="ETH",
    timeframe="1D"
)

In [ ]:
save_experiment_summary(
    df=btc,
    strategy_name="pvt_mdx",
    asset="ETH",
    timeframe="4H"
)